# Hull Tactical - Market Prediction: 高スコアnotebook解説

- **コンペ**: [Hull Tactical - Market Prediction](https://www.kaggle.com/competitions/hull-tactical-market-prediction)（S&P500の超過リターンを予測し、ボラティリティ制約下で市場に勝つ配分戦略を作る、実務寄りの金融コンペ。賞金総額$100,000）
- **元notebook**: [HTMP-OPTUNA-V8-FINAL-CPU](https://www.kaggle.com/code/frankmorales/htmp-optuna-v8-final-cpu) by **frankmorales**（Public LB Score **1.053**、51 votes、Silver medal）
- **手法概要**: Polarsで大量の時系列特徴量（ラグ・移動窓統計・ランク・Zスコア・交互作用項、最終的に1187列）を作り、XGBoost・LightGBM・CatBoostの3モデルアンサンブルを構築。Optunaによるハイパーパラメータ探索（3000試行以上）で得られたTrial 3258の設定（重み・ハイパーパラメータ）を最終版として採用している。
- **お断り**: これは学習目的の解説付き写しです。コード自体は改変していませんが、Kaggle上のHTML表示からテキスト取得した際にインデント情報が失われたため、字下げは元のロジックが破綻しないよう書き起こし時に再構成しています。出力（実行結果）はコピー元に含めていません。notebook内には内容がほぼ同一のセルが2つ（v4版・v5版、リスク係数の扱いのみ僅かに異なる）ありましたが、重複を避けるため最終版（v5、実際にKaggle Score 1.053を出した設定）のみを収録しています。

## 評価指標

- **タスク**: 毎営業日、S&P500への資金配分比率（0〜2倍、レバレッジ込み）を予測し、市場（S&P500）に対して超過リターンを狙いつつ、過度なボラティリティを取らない「マーケットタイミング」戦略を構築する。
- **指標**: notebook内の`score`関数に実装されている、**Sharpe比（シャープレシオ）にボラティリティ・ペナルティとリターン未達ペナルティを掛けた調整後Sharpe比**。具体的には、戦略の年率ボラティリティが市場ボラティリティの1.2倍を超えた分だけペナルティを掛け（`vol_penalty`）、さらに戦略の平均リターンが市場平均リターンを下回った場合は超過分の2乗に比例するペナルティを掛ける（`return_penalty`）設計。
- **なぜこの指標か**: 単純なリターンの大小だけを競うと、ハイレバレッジで博打的に張った戦略が有利になってしまう。Sharpe比（リターン÷リスク）をベースにしつつ、「市場より大幅にリスクを取りすぎていないか」「市場に負けていないか」を両方チェックすることで、**リスク調整後に安定して市場を上回る**戦略を評価しようという設計思想が読み取れる（初心者向け補足: Sharpe比は「リスク1単位あたりどれだけリターンを得られたか」を示す、金融でもっとも標準的なパフォーマンス指標の1つ）。
- **この手法がどう指標を最適化しているか**: 予測を「翌日プラスかマイナスか」の二値分類問題に落とし込み（`target_binary`）、3モデルの予測確率を加重平均した「自信度」（0.5からの距離）に`ML_CONF_FACTOR`という係数を掛けて0〜2の配分に変換している。つまり「自信があるときは大きく張り、自信がないときは市場並みに投資する（配分を1に近づける）」設計で、これはボラティリティペナルティを避けつつリターンを狙う、この指標の構造を意識した戦略設計になっている。Optunaでこの`ML_CONF_FACTOR`や各モデルの重み・ハイパーパラメータを直接この指標に対して探索している点が最大の特徴。


### Optunaハイパーパラメータ探索の要約（著者の考察）

Trial 3258が「ローカル検証で頑健性が最も高いモデル」として選ばれている。以下は探索過程で著者がまとめた比較表（要約）。

| Trial | ローカル検証スコア | Kaggleスコア | 汎化ギャップ(Kaggle-検証) | 備考 |
|---|---|---|---|---|
| 397（過去最高） | 1.01052 | 1.112 | +0.10148 | 運が良かっただけの不安定な汎化 |
| 957 | 1.02287 | N/A | N/A | 検証スコアは397より高い |
| 3258 | **1.06937** | **1.053** | **-0.01637** | **最も安定した高得点（低下幅が最小）** |
| 3319 | 1.03807 | 0.994 | -0.04407 | 3258よりオーバーフィット |

著者の結論：検証スコアの高さだけでなく「本番でどれだけスコアが落ちるか（汎化ギャップ）」を重視し、**低下幅が最小のTrial 3258を最終採用**している。特徴量セットも当初1187列から拡張を試したが、ノイズが増えてスコアが0.92まで悪化したため、元の1187列セットに戻したという記録も残されている。


### 採用されたアンサンブルの重みとハイパーパラメータ（要約）

3モデルアンサンブル（XGBoost・LightGBM・CatBoost）の中で、**XGBoostが最も重み付けが大きい（4.18）主力モデル**、CatBoostが次点（1.93）、LightGBMは最も小さい重み（0.30、多様性確保のための「味付け」的役割）という構成。

- **XGBoost**（weight 4.18）: `max_depth=14`（かなり深い木で複雑な非線形相互作用を捉える）, `learning_rate=0.0326`（保守的な学習率）, L1/L2正則化あり, `subsample=0.80` / `colsample=0.73`（過学習防止のための行・列のサブサンプリング）。目的関数は`binary:logistic`、`tree_method='hist'`（ヒストグラムベースの高速アルゴリズム）。学習は最大5000本まで許すが、Early Stopping（50ラウンド改善なしで停止）により実際は18〜22本で停止。
- **LightGBM**（weight 0.30、最小）: `max_depth=8`（浅め）, `num_leaves=141`（デフォルト31よりかなり多く複雑な木構造）, 小さな学習率(0.014)。GPUを使用。
- **CatBoost**（weight 1.93）: `depth=11`, `bootstrap_type='Bayesian'`（重みのサンプリング方法の一種で、ターゲットリークを防ぐCatBoost独自の手法）, GPU使用。

3モデルとも「深い木＋強い正則化＋低い学習率」という、金融の時系列データにありがちな過学習リスクを意識した保守的な設定になっているのが特徴です（初心者向け補足: 学習率を下げて木の本数を増やすほど、一般に汎化性能は安定しやすいが学習時間は伸びるというトレードオフがある）。


In [ ]:
# FINAL VERSION - STABILIZED ENSEMBLE WITH OPTUNA FACTOR (v5)
import os
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import pandas.api.types
from itertools import product
import warnings
import logging
import lightgbm as lgb
import platform
import sys
from contextlib import contextmanager

# --- Setup and Logging ---
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', module='lightgbm')

os.environ['CATBOOST_DRIVER_COMPATIBLE'] = '1'
os.environ['CATBOOST_QUIET_MODE'] = '1'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')


# ===========================================================================
# WARNING SUPPRESSION CONTEXT MANAGER
# ===========================================================================
@contextmanager
def suppress_stderr():
    """Temporarily redirect stderr to devnull to suppress native C warnings."""
    original_stderr = sys.stderr
    try:
        with open(os.devnull, 'w') as f:
            sys.stderr = f
        yield
    finally:
        sys.stderr = original_stderr


# ===========================================================================
# OFFICIAL KAGGLE METRIC
# ===========================================================================
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = 'row_id') -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ValueError('Predictions must be numeric')

    sol = solution.copy()
    sol['position'] = submission['prediction'].values

    if sol['position'].max() > MAX_INVESTMENT:
        raise ValueError(f'Position exceeds {MAX_INVESTMENT}')
    if sol['position'].min() < MIN_INVESTMENT:
        raise ValueError(f'Position below {MIN_INVESTMENT}')

    sol['strategy_returns'] = sol['risk_free_rate'] * (1 - sol['position']) + sol['position'] * sol['forward_returns']
    strategy_excess = sol['strategy_returns'] - sol['risk_free_rate']
    strategy_cum = (1 + strategy_excess).prod()
    strategy_mean = strategy_cum ** (1 / len(sol)) - 1
    strategy_std = sol['strategy_returns'].std()
    trading_days = 252

    if strategy_std == 0:
        return 0.0
    sharpe = strategy_mean / strategy_std * np.sqrt(trading_days)

    strategy_vol = float(strategy_std * np.sqrt(trading_days) * 100)

    market_excess = sol['forward_returns'] - sol['risk_free_rate']
    market_cum = (1 + market_excess).prod()
    market_mean = market_cum ** (1 / len(sol)) - 1
    market_std = sol['forward_returns'].std()
    market_vol = float(market_std * np.sqrt(trading_days) * 100)

    if market_vol == 0:
        return 0.0

    excess_vol = max(0, strategy_vol / market_vol - 1.2)
    vol_penalty = 1 + excess_vol

    return_gap = max(0, (market_mean - strategy_mean) * 100 * trading_days)
    return_penalty = 1 + (return_gap ** 2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


**何をしているか**: 環境セットアップと、コンペ公式のスコア計算関数`score`をnotebook内に再現している。
**なぜそうするのか**: `score`関数を手元に再現しておくことで、本番提出前にローカルの検証データでスコアを試算できる（コンペ側の非公開の採点コードを信頼するだけでなく、手元で同じロジックを実装して検証する、という堅実なやり方）。中身を読むと、`vol_penalty`（ボラティリティが市場の1.2倍を超えた分だけ罰則）と`return_penalty`（市場平均リターンに届かない分の2乗に比例した罰則）という2つの罰則項でSharpe比を割っており、「リスクを取りすぎず、かつ市場に負けない」ことを両立させる設計だと分かる（初心者向け補足: 罰則項で割ることで、単純にリターンやリスクだけを見るより「バランスの良い戦略」が高スコアになるよう誘導している）。

In [ ]:
# ===========================================================================
# FEATURE ENGINEERING FUNCTION (POLARS - RESTORED ORIGINAL SET)
# ===========================================================================
def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df_copy = df.clone()
    potential_base_feature_prefixes = ('M', 'E', 'I', 'P', 'V', 'S')
    all_potential_features = [
        c for c in df_copy.columns
        if c.startswith(potential_base_feature_prefixes) and c != 'market_forward_excess_returns'
    ]
    casting_expressions = []
    for c in all_potential_features:
        casting_expressions.append(pl.col(c).cast(pl.Float64, strict=False).alias(c))
    if casting_expressions:
        df_copy = df_copy.with_columns(casting_expressions)
    base_features = [c for c in all_potential_features if df_copy.schema.get(c) in pl.NUMERIC_DTYPES]
    expressions = []

    # --- 1. Lags (Original: 4 Lags) ---
    for c in base_features:
        for lag in [1, 2, 5, 10]:
            expressions.append(
                pl.col(c).shift(lag).over('date_id').fill_null(0).alias(f'{c}_L{lag}')
            )

    # --- 2. Rolling Window Features (Original: 2 Windows, 3 Stats) ---
    for c in base_features:
        for w in [5, 10]:
            expressions.append(
                pl.col(c).rolling_mean(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMean{w}')
            )
            expressions.append(
                pl.col(c).rolling_std(window_size=w, min_periods=1).over('date_id').fill_nan(0).fill_null(0).alias(f'{c}_RStd{w}')
            )
            expressions.append(
                pl.col(c).rolling_max(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMax{w}')
            )

    df_copy = df_copy.with_columns(expressions)
    expressions = []

    # --- 3. Rank and Z-Score ---
    for c in base_features:
        expressions.append(
            pl.col(c).rank(method='min').over('date_id').fill_null(0).alias(f'{c}_RANK')
        )
        mean_c = pl.col(c).mean().over('date_id')
        std_c = pl.col(c).std().over('date_id')
        std_c_safe_expr = pl.when(pl.col(c).is_null() | std_c.is_null() | (std_c == 0)).then(1e-6).otherwise(std_c)
        expressions.append(
            ((pl.col(c).fill_null(mean_c) - mean_c) / std_c_safe_expr).fill_nan(0).fill_null(0).alias(f'{c}_ZSCORE')
        )
    df_copy = df_copy.with_columns(expressions)
    rank_cols = [f'{c}_RANK' for c in base_features]
    zscore_cols = [f'{c}_ZSCORE' for c in base_features]
    expressions = []

    # --- 4. Interactions (Original Set only) ---
    # Original Targeted Interactions (M4, M1, E1, V2)
    for c in ['M4', 'M1', 'E1', 'V2']:
        r = f'{c}_RANK'
        if c in df_copy.columns and r in df_copy.columns:
            expressions.append((pl.col(c) * pl.col(r)).alias(f'{c}_x_{r}'))
            expressions.append((pl.col(c) / (pl.col(r) + 1e-6)).alias(f'{c}_div_{r}'))

    target_rank_cols = rank_cols[:12]
    # 1. Rank * Rank (Unique Pairs) - Adds 66 features
    for r_col1, r_col2 in product(target_rank_cols, target_rank_cols):
        c1 = r_col1.split('_')[0]
        c2 = r_col2.split('_')[0]
        if c1 < c2:
            expressions.append((pl.col(r_col1) * pl.col(r_col2)).fill_nan(0).fill_null(0).alias(f'{c1}R_x_{c2}R'))
    # 2. Rank * ZScore - Adds 9 features
    target_zscore_cols = zscore_cols[:9]
    for r_col, z_col in zip(rank_cols[:9], target_zscore_cols):
        expressions.append((pl.col(r_col) * pl.col(z_col)).fill_nan(0).fill_null(0).alias(f'{r_col}_x_{z_col}'))

    if expressions:
        df_copy = df_copy.with_columns(expressions)
    return df_copy


**何をしているか**: `M`（マクロ経済）・`E`（Eコマース系？）・`I`・`P`・`V`（ボラティリティ関連）・`S`（センチメント？）で始まる元の数値列それぞれについて、(1) 1/2/5/10日前のラグ、(2) 5日・10日の移動平均・移動標準偏差・移動最大値、(3) 日内（`date_id`ごと）の順位（rank）とZスコア、(4) 主要4列×ランクの交互作用と、上位12列同士のランク×ランク交互作用（66通り）を作る大規模な特徴量エンジニアリング関数。最終的に1187列に膨れ上がる。
**なぜそうするのか**: 金融時系列では「今日の値」そのものより「直近との変化」「他の銘柄・指標との相対順位」の方が予測に効くことが多い（初心者向け補足: ラグ特徴量は「過去の値を新しい列として持たせる」ことでモデルに時系列の記憶を与える基本テクニック、移動窓統計量はノイズを均して傾向を捉えるためのテクニック）。`.over('date_id')`はPolarsの「グループごとの計算」機能で、日付ごとにグループ化してランクや統計量を計算している（初心者向け補足: pandasの`groupby().transform()`に近い操作）。ランク×ランクの交互作用を上位12列に絞っているのは、全列の組み合わせを作ると計算量が爆発するため、重要そうな列に絞る現実的な妥協。

In [ ]:
# ===========================================================================
# DATA LOADING AND SPLITTING
# ===========================================================================
logger.info("Loading data and splitting for validation...")
try:
    train_full_pl = pl.read_csv(DATA_PATH / "train.csv")
except FileNotFoundError:
    logger.error("Could not find 'train.csv'. Please ensure the DATA_PATH is correct.")
    raise

train_full_pd = train_full_pl.to_pandas()

split_idx = int(len(train_full_pd) * 0.8)
train_pd = train_full_pd.head(split_idx).copy()
val_pd = train_full_pd.tail(len(train_full_pd) - split_idx).copy()
full_train_pd = train_full_pd.copy()

# --- Revert to Classification Target (Required for Optimal Score) ---
train_pd['target_binary'] = (train_pd['market_forward_excess_returns'] > 0).astype(np.int8)
val_pd['target_binary'] = (val_pd['market_forward_excess_returns'] > 0).astype(np.int8)
full_train_pd['target_binary'] = (full_train_pd['market_forward_excess_returns'] > 0).astype(np.int8)

train_pl = pl.from_pandas(train_pd)
val_pl = pl.from_pandas(val_pd)
full_train_pl = pl.from_pandas(full_train_pd)

logger.info("Oracle Dictionary ready. Binary Target created.")

# ===========================================================================
# ENSEMBLE PARAMETERS (OPTUNA TRIAL 3258 BEST, Score 1.06937)
# ===========================================================================
BEST_W_XGB = 4.183022496349415       # XGBoost Weight
BEST_W_LGB = 0.29640061148676144     # LightGBM Weight
BEST_W_CAT = 1.9283230458964578      # CatBoost Weight
BEST_W_LOGREG = 0.0
ML_CONF_FACTOR = 6.6208527134892385  # Optimal Risk Factor

# Model-specific Hyperparameters (Trial 3258)
XGB_MAX_DEPTH = 14
XGB_LR = 0.03260054929298955
XGB_REG_ALPHA = 0.1722628424051081
XGB_REG_LAMBDA = 10.528253815338909
XGB_SUBSAMPLE = 0.8018511121220447
XGB_COLSAMPLE = 0.7282877621162328

LGB_MAX_DEPTH = 8
LGB_LR = 0.013960942809215957
LGB_NUM_LEAVES = 141
LGB_L1 = 0.9245109459111609

CBT_DEPTH = 11
CBT_LR = 0.03679836629440696
CBT_L2_REG = 4.601313084093791

# Fixed Estimators and Stopping
N_ESTIMATORS_MAX = 5000
XGB_ES = 200               # Use a moderate patience for the dominant model
LGB_CAT_ES = N_ESTIMATORS_MAX  # Ensure full train for unstable models

logger.info(f"Using OPTUNA BEST ENSEMBLE (Trial 3258), Weights: XGB:{BEST_W_XGB:.2f}, LGB:{BEST_W_LGB:.2f}, CAT:{BEST_W_CAT:.2f}")
logger.info(f"Using RESTORED Optuna Risk Factor: {ML_CONF_FACTOR:.4f}")


**何をしているか**: 学習データを時系列順に先頭80%（train）・末尾20%（val）に分割し、目的変数を「翌日の超過リターンがプラスかどうか」の二値に変換する。あわせて、Optuna Trial 3258で見つかった最終的な重み・ハイパーパラメータを定数として定義する。
**なぜそうするのか**: 時系列データなのでランダムに分割せず、**先頭を学習・末尾を検証**にする時系列split（未来のデータで過去を予測してしまう「時間方向のリーク」を防ぐ基本原則）を使っている。回帰（実際のリターン値を当てる）ではなく二値分類にしているのは、後段で「上がる確信度」を配分量に変換する設計にするため（初心者向け補足: 金融の実務でも「量」より「方向」を当てる方が簡単で安定しやすいとされることが多い）。

In [ ]:
# ===========================================================================
# ENSEMBLE PIPELINE (MAIN PIPELINE)
# ===========================================================================
logger.info("Training ensemble (Expanded Feature Set)...")

# 1. Feature Engineering
train_engineered_pl = create_features(train_pl)
val_engineered_pl = create_features(val_pl)
full_train_engineered_pl = create_features(full_train_pl)

# 2. Feature Column Selection
feature_cols = []
original_suffixes = ('_L1', '_L2', '_L5', '_L10', '_RMean5', '_RStd5', '_RMax5',
                      '_RMean10', '_RStd10', '_RMax10', '_RANK', '_ZSCORE',
                      '_x_RANK', '_div_RANK', '_x_R', '_x_ZSCORE')

for col in train_engineered_pl.columns:
    is_base_feature = col.startswith(('M', 'E', 'I', 'P', 'V', 'S'))
    is_engineered_feature = any(col.endswith(s) for s in original_suffixes)
    if (is_base_feature or is_engineered_feature) and (
        train_engineered_pl[col].null_count() / len(train_engineered_pl) < 0.95
    ):
        feature_cols.append(col)

feature_cols = [c for c in feature_cols
                if c not in ['market_forward_excess_returns', 'target_binary', 'forward_returns', 'date_id']]
FINAL_FEATURE_COLS = feature_cols
logger.info(f"Features: {len(FINAL_FEATURE_COLS)} (Restored Optimized Set)")

# 3. Data Preparation for ML
# XGBoost Data (80% train, 20% val for ES)
X_train = train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_train = train_pl['target_binary'].fill_null(0).to_numpy().astype(int)
X_val = val_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_val = val_pl['target_binary'].fill_null(0).to_numpy().astype(int)

# LGB/CAT Data (100% full train set)
X_full_train = full_train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_full_train = full_train_pl['target_binary'].fill_null(0).to_numpy().astype(int)

# Enforce Feature Order
X_train = X_train.loc[:, FINAL_FEATURE_COLS]
X_val = X_val.loc[:, FINAL_FEATURE_COLS]
X_full_train = X_full_train.loc[:, FINAL_FEATURE_COLS]

# Scaling (Fit only on 80% train data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_full_train_scaled = scaler.transform(X_full_train)

# DataFrames for LGB and DMatrix for XGB
X_lgb = pd.DataFrame(X_train_scaled, columns=FINAL_FEATURE_COLS)
X_val_lgb = pd.DataFrame(X_val_scaled, columns=FINAL_FEATURE_COLS)
X_full_lgb = pd.DataFrame(X_full_train_scaled, columns=FINAL_FEATURE_COLS)

dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dval = xgb.DMatrix(X_val_scaled, label=y_val)


**何をしているか**: 生成した特徴量のうち、欠損率95%未満の列だけを選んで最終的な特徴量セット（`FINAL_FEATURE_COLS`）とし、80%学習データでスケーラー（標準化：平均0・分散1に揃える前処理）を学習してtrain/val/full setすべてに適用する。XGBoost用には`DMatrix`（XGBoost独自の高速なデータ構造）も作成。
**なぜそうするのか**: 欠損率95%以上の列は情報量がほとんどなく、学習の邪魔になるだけなので機械的に除外している。スケーラーを**80%訓練データだけでfitし、val/full setにはtransformだけ適用する**のは、検証データの統計量が学習に漏れ込む「リーク」を防ぐための鉄則（初心者向け補足: `fit`は「学習データから平均・分散などの統計量を計算する」処理、`transform`は「その統計量を使って変換だけする」処理。検証・テストデータでは絶対に`fit`し直してはいけない）。

In [ ]:
# 4. Model Training (3 Models + 1 inert model)
with suppress_stderr():
    # --- XGBoost (Trained on 80% data with ES) ---
    xgb_params = {
        'max_depth': XGB_MAX_DEPTH, 'eta': XGB_LR, 'alpha': XGB_REG_ALPHA, 'lambda': XGB_REG_LAMBDA,
        'subsample': XGB_SUBSAMPLE, 'colsample_bytree': XGB_COLSAMPLE, 'seed': 42,
        'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss', 'verbosity': 0,
    }

    xgb_model = xgb.train(
        xgb_params, dtrain, N_ESTIMATORS_MAX, evals=[(dval, 'val')],
        early_stopping_rounds=XGB_ES, verbose_eval=False,
    )
    logger.info(f"XGBoost trained (Stopped at {xgb_model.best_iteration} trees).")

    # --- LightGBM (Trained on 100% data, NO ES/VALIDATION CHECK) ---
    lgb_model = LGBMClassifier(
        n_estimators=N_ESTIMATORS_MAX, max_depth=LGB_MAX_DEPTH, learning_rate=LGB_LR,
        num_leaves=int(LGB_NUM_LEAVES), lambda_l1=LGB_L1, random_state=123,
        objective='binary', metric='binary_logloss', n_jobs=-1, device_type='gpu',
        verbose=-1, _log_period=-1,
    ).fit(X_full_lgb, y_full_train)
    logger.info(f"LGBMClassifier trained (Full 100% data, {lgb_model.n_estimators} trees).")

    # --- CatBoost (Trained on 100% data, NO ES/VALIDATION CHECK) ---
    cbt_model = CatBoostClassifier(
        iterations=N_ESTIMATORS_MAX, depth=int(CBT_DEPTH), learning_rate=CBT_LR,
        l2_leaf_reg=CBT_L2_REG, random_seed=789, loss_function='Logloss', eval_metric='Logloss',
        bootstrap_type='Bayesian', task_type="GPU", logging_level='Silent', allow_writing_files=False,
    ).fit(X_full_train, y_full_train)
    logger.info(f"CatBoostClassifier trained (Full 100% data, {cbt_model.get_best_iteration()} iterations).")

    # --- Logistic Regression (Trained on 80% data, but weighted 0.0) ---
    logreg_model = LogisticRegression(
        penalty='l2', C=0.01, solver='liblinear', max_iter=1000, random_state=999,
    ).fit(X_lgb, y_train)
    logger.info("LogisticRegression trained.")

logger.info("Ensemble trained.")


**何をしているか**: XGBoost・LightGBM・CatBoost・ロジスティック回帰（重み0で実質不使用、比較用の「保険」として残している）の4モデルを学習する。XGBoostだけ80%訓練データ＋Early Stopping、LightGBM/CatBoostは全データで固定本数まで学習という**非対称な学習方針**になっている。
**なぜそうするのか**: XGBoostは主力モデル（重み4.18）なので、検証データでEarly Stoppingをかけて過学習を防ぐ慎重な設計。一方LightGBM/CatBoostは重みが小さく「多様性を加える脇役」という位置づけのためか、全データで学習させて（検証によるチェックなしで）様子見的に使っている、という判断が読み取れる。ロジスティック回帰は重み0.0で最終予測には影響しないが、コードには残されている——これはOptuna探索の過程で「線形モデルも試したが今回は不要だった」という試行錯誤の痕跡だと考えられる。

In [ ]:
# ===========================================================================
# VALIDATION - SCORE INTEGRATION (3-MODEL ENSEMBLE)
# ===========================================================================
logger.info("Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...")

# XGBoostの予測はEarly Stoppingで確定したbest_iterationまでを使い、
# LGB/CatBoostは全5000イテレーションのモデルをそのまま使う。
prob_xgb = xgb_model.predict(dval, iteration_range=(0, xgb_model.best_iteration))
prob_lgb = lgb_model.predict_proba(X_val_lgb)[:, 1]
prob_cbt = cbt_model.predict_proba(X_val)[:, 1]
prob_logreg = logreg_model.predict_proba(X_val_lgb)[:, 1]

# 3モデルの重み付き平均で最終的な「上がる確率」を算出。
total_w_final = BEST_W_XGB + BEST_W_LGB + BEST_W_CAT + BEST_W_LOGREG
avg_prob = (BEST_W_XGB * prob_xgb + BEST_W_LGB * prob_lgb + BEST_W_CAT * prob_cbt
            + BEST_W_LOGREG * prob_logreg) / total_w_final

confidence = 2 * np.abs(avg_prob - 0.5)
positions_final = np.clip(confidence * ML_CONF_FACTOR, 0.0, 2.0)

submission_df_final = pd.DataFrame({'prediction': positions_final})

try:
    real_ps = score(val_pd, submission_df_final)
    logger.info(f"FINAL SUBMISSION PS SCORE ON VALIDATION (STABILIZED, OPTUNA FACTOR) = {real_ps:.6f}")
except Exception as e:
    logger.error(f"Final Scoring error: {e}")
    real_ps = 0.0


**何をしているか**: 検証データで3モデルの予測確率を重み付き平均し、`confidence = 2 * |avg_prob - 0.5|`（0.5からどれだけ離れているか＝自信度、0〜1に正規化）を計算。これに`ML_CONF_FACTOR`（Optunaで探索したリスク係数）を掛けて0〜2の配分量に変換し、手元実装の`score`関数で試算する。
**なぜそうするのか**: これがこの手法の核心部分。予測確率をそのまま配分量にするのではなく、「0.5から離れているほど自信がある」という発想で配分量を決めている（初心者向け補足: 予測確率0.5＝五分五分の予想。0.9や0.1のように0.5から離れているほど「自信がある」とみなせる）。`ML_CONF_FACTOR`という1つの係数で「自信度をどれだけ大胆な配分に変換するか」を制御しており、これをOptunaで指標（Sharpe比ベースのスコア）に対して直接最適化しているのが、単なる「精度の高い分類器を作る」以上の工夫。

In [ ]:
# ===========================================================================
# PREDICT FUNCTION (FINAL ROBUST VERSION - 3-MODEL ENSEMBLE)
# ===========================================================================
def predict(test: pl.DataFrame) -> float:
    # predict関数のスコープ内で定数を再定義（推論サーバーから呼ばれる際に外側の変数に
    # 依存しすぎないようにするための防御的な書き方）。
    BEST_W_XGB = 4.183022496349415
    BEST_W_LGB = 0.29640061148676144
    BEST_W_CAT = 1.9283230458964578
    BEST_W_LOGREG = 0.0
    ML_CONF_FACTOR = 6.6208527134892385

    if (xgb_model is None or lgb_model is None or cbt_model is None or logreg_model is None
            or scaler is None or not FINAL_FEATURE_COLS):
        return 0.0

    date_id = None
    try:
        date_id = int(test.select("date_id").to_series().item())
    except Exception:
        pass

    try:
        test_engineered = create_features(test)
        X_test = test_engineered.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()

        X_clean = X_test.loc[:, FINAL_FEATURE_COLS]
        X_clean = X_clean.fillna(0).replace([np.inf, -np.inf], 0)

        X_scaled = scaler.transform(X_clean.values)
        X_lgb = pd.DataFrame(X_scaled, columns=FINAL_FEATURE_COLS)
        dtest = xgb.DMatrix(X_scaled)

        prob_xgb = xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration))[0]
        prob_lgb = lgb_model.predict_proba(X_lgb)[:, 1][0]
        prob_cbt = cbt_model.predict_proba(X_clean)[:, 1][0]
        prob_logreg = logreg_model.predict_proba(X_lgb)[:, 1][0]

        total_w = BEST_W_XGB + BEST_W_LGB + BEST_W_CAT + BEST_W_LOGREG
        avg_prob = (BEST_W_XGB * prob_xgb + BEST_W_LGB * prob_lgb + BEST_W_CAT * prob_cbt
                    + BEST_W_LOGREG * prob_logreg) / total_w

        confidence = 2 * abs(avg_prob - 0.5)
        position = np.clip(confidence * ML_CONF_FACTOR, 0, 2)
        return float(position)

    except Exception as e:
        logger.warning(f"ML Error (date_id: {date_id}): {e}")
        return 0.0


# ===========================================================================
# SERVER
# ===========================================================================
logger.info("Starting server")
import kaggle_evaluation.default_inference_server
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

logger.info("Complete.")


**何をしているか**: 本番の逐次評価用に、1日分のデータを受け取って配分量を1つ返す`predict`関数を定義し、`kaggle_evaluation`ライブラリの推論サーバーに登録する。`except Exception`で失敗時は`0.0`（無配分＝ノーポジション）を返すようフォールバックしている。
**なぜそうするのか**: このコンペは「Code Competition」で、モデルは日々のデータを受け取って毎回配分を返す**API形式のサーバー**として動作する（`kaggle_evaluation`はKaggleが提供する評価用ハーネス）。もしどこかの日で予測処理がエラーになった場合、サーバーごと停止してしまうと以降すべての日が採点不能になるため、**エラー時は保守的に「投資しない(0.0)」を返して処理を継続させる**というフェイルセーフ設計になっている（初心者向け補足: 本番運用のシステムでは「エラーで全部止まる」より「安全側に倒して継続する」設計が好まれることが多い、という実務的な教訓）。

In [ ]:
import pandas as pd
import numpy as np
import os

submission_path = '/kaggle/working/submission.parquet'

# 提出ファイルが正しく作られているかの最終チェック。
if not os.path.exists(submission_path):
    print(f"Validation Error: Submission file not found at {submission_path}")
    print("NOTE: This file is created during the 'Running local gateway for testing...' step.")
else:
    try:
        df_sub = pd.read_parquet(submission_path)

        float_cols = df_sub.select_dtypes(include=[np.float64]).columns
        if len(float_cols) == 1:
            prediction_col_name = float_cols[0]
            print(f"Column Check: Found single prediction column named '{prediction_col_name}'.")
        else:
            prediction_col_name = 'allocation'
            print(f"Column Check: Found {len(float_cols)} float columns. Using '{prediction_col_name}' for range check.")

        if prediction_col_name in df_sub.columns:
            min_val = df_sub[prediction_col_name].min()
            max_val = df_sub[prediction_col_name].max()
            if min_val >= 0.0 and max_val <= 2.0:
                range_check = "PASS"
            else:
                range_check = f"FAIL (Min: {min_val:.4f}, Max: {max_val:.4f})"
            print(f"Allocation Range Check (0.0 to 2.0): {range_check}")

            print("\nFirst 5 Rows of Submission:")
            print(df_sub.head())
            print("\nSubmission Info:")
            df_sub.info()

    except Exception as e:
        print(f"Validation Error: Could not read or process the Parquet file. Error: {e}")


**何をしているか**: 生成された提出ファイル（`submission.parquet`）を読み込み、配分量が規約通り0.0〜2.0の範囲に収まっているかを機械的にチェックする、提出前の最終バリデーション。
**なぜそうするのか**: コンペのルール上、配分量が範囲外だとエラー扱いになる（`score`関数内でも`ValueError`を出す実装だった）。提出直前にプログラムでレンジチェックをかけておくことで、うっかりルール違反の提出をしてしまうケアレスミスを防いでいる（初心者向け補足: 機械学習コンペに限らず、本番投入前の「サニティチェック」は地味だが事故を防ぐ重要な工程）。

## まとめ

このnotebookは、大規模な時系列特徴量エンジニアリング（Polarsによる1187列生成）とXGBoost/LightGBM/CatBoostの3モデルアンサンブルという「王道」の構成に加え、**Optunaで数千試行のハイパーパラメータ探索を行い、検証スコアの高さだけでなく「本番でのスコア低下幅（汎化ギャップ）が最小のTrial」を選ぶ**という慎重な採用基準が学びどころです。また、予測確率をそのまま使うのではなく「0.5からの距離＝自信度」を配分量に変換するロジックと、それを制御する`ML_CONF_FACTOR`を指標に対して直接最適化している点は、単純な分類精度の最大化とは異なる、**評価指標そのものを見据えたモデル設計**の好例と言えます。